# 01 — Spark Core Concepts

Spark architecture, RDDs vs DataFrames, lazy evaluation, and the job/stage/task execution model — the vocabulary every data engineer interview opens with.

> **Setup note:** these notebooks are written but **not executed** — PySpark is not
> installed in this environment. To run them locally:
>
> ```bash
> python -m venv .venv && source .venv/bin/activate
> pip install pyspark==3.5.1
> # Java 11/17 must be on PATH (java -version)
> jupyter notebook
> ```
>
> Everything below is correct, runnable PySpark — read it as a reference and run
> cell-by-cell once your environment is set up.

## 1. Architecture: Driver, Executors, Cluster Manager

- **Driver**: the process running your `main()` / notebook. It builds the logical plan (DAG), negotiates resources with the cluster manager, and schedules tasks on executors. If the driver dies, the whole application dies.
- **Executors**: JVM processes on worker nodes that actually run tasks and hold cached data in memory/disk. Each executor has a fixed number of cores (= max concurrent tasks) and a memory budget.
- **Cluster manager**: allocates resources to the Spark application — YARN, Kubernetes, Mesos, or Spark's own Standalone manager. In `local[*]` mode the driver and executors are threads in one JVM (used for dev/testing).
- **SparkContext / SparkSession**: the entry point. `SparkSession` (Spark 2.0+) wraps `SparkContext`, `SQLContext`, and `HiveContext` into one object.

**Interview framing:** "Spark distributes a computation by splitting data into partitions, running the same task in parallel across executors, one task per partition per stage."

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("spark-core-concepts")
    .master("local[*]")          # all local cores; use a cluster URL in prod
    .config("spark.sql.shuffle.partitions", "8")  # small for local demos
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
spark

## 2. RDD vs DataFrame vs Dataset

| | RDD | DataFrame | Dataset |
|---|---|---|---|
| Type safety | compile-time (generic `T`) | none (rows are untyped `Row`) | compile-time (JVM only, not in PySpark) |
| Optimization | none — you write the execution plan | Catalyst optimizer + Tungsten | Catalyst + Tungsten |
| API level | low-level, functional (`map`, `filter`, `reduce`) | declarative, SQL-like | hybrid |
| When to use | fine-grained control, unstructured data, custom partitioning | 95% of real workloads — structured/semi-structured data | Scala/Java only |

**Key interview point:** PySpark does *not* have Datasets (Python is dynamically typed) — in PySpark it's RDDs and DataFrames. DataFrames are preferred by default because Catalyst can optimize the plan (predicate pushdown, column pruning, join reordering); RDDs are a black box to the optimizer.

In [ ]:
# RDD: low-level, functional API
rdd = spark.sparkContext.parallelize(range(1, 11), numSlices=4)
squared_evens = rdd.filter(lambda x: x % 2 == 0).map(lambda x: x * x)
print(squared_evens.collect())      # [4, 16, 36, 64, 100]
print(rdd.getNumPartitions())        # 4

In [ ]:
# The same logic as a DataFrame — declarative, optimizable
from pyspark.sql.functions import col

df = spark.range(1, 11).toDF("n")
result = df.filter(col("n") % 2 == 0).withColumn("squared", col("n") * col("n"))
result.show()

## 3. Transformations vs Actions — Lazy Evaluation

- **Transformations** (`select`, `filter`, `map`, `join`, `groupBy`, ...) are **lazy**: they just build up a logical plan (a DAG of operations). Nothing runs.
- **Actions** (`collect`, `count`, `show`, `write`, `take`, `foreach`, ...) trigger execution: Spark optimizes the accumulated plan and runs it.
- Why lazy? It lets Catalyst see the *whole* pipeline before running anything, so it can reorder filters before joins, prune unused columns, push predicates down to the data source, etc. Eager execution (like pandas) can't do this.

**Execution hierarchy:** Application → Jobs (one per action) → Stages (split at shuffle boundaries) → Tasks (one per partition, the unit that actually runs on an executor core).

In [ ]:
# Nothing executes here — just building a plan
lazy_plan = df.filter(col("n") > 3).withColumn("squared", col("n") ** 2)
print(type(lazy_plan))  # still just a DataFrame, no computation has happened

# This action triggers a job: Spark now optimizes + runs the DAG
lazy_plan.show(3)

In [ ]:
# .explain() shows the plan without an action forcing a *result*
# (explain itself does trigger plan analysis, but not a data-scanning job)
lazy_plan.explain(mode="formatted")  # shows Parsed -> Analyzed -> Optimized -> Physical plan

## 4. Common interview questions — Section 1

1. **"What happens when you call `.collect()` on a 1 TB DataFrame?"** — it pulls *all* partitions to the driver's memory. Almost always wrong in production; use `.write`, `.take(n)`, or aggregate first.
2. **"Why is Spark lazy?"** — enables whole-pipeline optimization via Catalyst (predicate pushdown, column pruning) instead of executing each line eagerly like pandas.
3. **"Difference between a job, a stage, and a task?"** — job = triggered by one action; stage = a chunk of the job's DAG that runs without a shuffle (shuffle boundaries split stages); task = the unit of work for one partition, run on one executor core.
4. **"Why prefer DataFrames over RDDs?"** — Catalyst/Tungsten optimizations, less code, SQL interop; drop to RDDs only for custom partitioning logic or truly unstructured data.
5. **"What's the difference between `local[*]`, client mode, and cluster mode?"** — `local[*]` runs everything in one JVM (dev only); client mode runs the driver on the machine that submitted the job (executors on the cluster); cluster mode runs the driver *inside* the cluster too (used for production jobs so the client can disconnect).

## Summary

- Driver plans and schedules; executors run tasks and hold cached data.
- Prefer DataFrames over RDDs for structured data — Catalyst optimizes them.
- Transformations are lazy; actions trigger a job. Job → stage (shuffle boundary) → task (per-partition).
- Next: `02_dataframes_io_and_schemas.ipynb` — reading/writing data and managing schemas correctly.